In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

model_name = "tabularisai/multilingual-sentiment-analysis"

print(f"Loading {model_name}...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# Pipeline approach
# sentiment_pipeline = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, device=0 if torch.cuda.is_available() else -1)

# print("\n--- Sentiment Analysis Example (Pipeline Way) ---")

# # Slovak test cases for pipeline, adapted for 'positive', 'negative', 'neutral' outputs
# text_positive_pipeline = "Táto reštaurácia je vynikajúca, jedlo bolo fantastické!" # Slovak positive
# prediction_positive_pipeline = sentiment_pipeline(text_positive_pipeline)
# print(f"\nInput: '{text_positive_pipeline}'")
# print(f"Prediction: {prediction_positive_pipeline}")

# text_negative_pipeline = "Služby boli strašné a personál bol neochotný." # Slovak negative
# prediction_negative_pipeline = sentiment_pipeline(text_negative_pipeline)
# print(f"\nInput: '{text_negative_pipeline}'")
# print(f"Prediction: {prediction_negative_pipeline}")

# text_neutral_pipeline = "Film bol v poriadku, ale nič výnimočné." # Slovak neutral
# prediction_neutral_pipeline = sentiment_pipeline(text_neutral_pipeline)
# print(f"\nInput: '{text_neutral_pipeline}'")
# print(f"Prediction: {prediction_neutral_pipeline}")

print("\n--- Manual Sentiment Analysis Example ---")

# Get label mapping from model config
id2label_mapping = model.config.id2label
print(f"Model's ID to Label mapping: {id2label_mapping}")


def get_manual_sentiment(text, model, tokenizer, device, label_map):
    # 1. Tokenize the input
    inputs = tokenizer(
        text,
        return_tensors="pt",
        add_special_tokens=True,
        truncation=True,
        padding='max_length',
        max_length=512
    ).to(device)

    # 2. Perform model inference
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits

    # 3. Apply softmax to get probabilities
    probabilities = torch.softmax(logits, dim=1)

    # 4. Get the predicted label and its score
    max_probability = torch.max(probabilities).item()
    predicted_class_id = torch.argmax(probabilities).item()
    # Use the label_map to get the human-readable label
    predicted_label = label_map[predicted_class_id]

    return {'label': predicted_label, 'score': max_probability}

# Example 1: Positive sentiment (Slovak - manual)
text_positive_manual = "Bol to skvelý zážitok, určite odporúčam každému!" # Slovak positive
prediction_positive_manual = get_manual_sentiment(text_positive_manual, model, tokenizer, device, id2label_mapping)
print(f"\nInput (manual way): '{text_positive_manual}'")
print(f"Prediction (manual way): {prediction_positive_manual}")

# Example 2: Negative sentiment (Slovak - manual)
text_negative_manual = "Úplne zlé, nikdy viac sa sem nevrátim." # Slovak negative
prediction_negative_manual = get_manual_sentiment(text_negative_manual, model, tokenizer, device, id2label_mapping)
print(f"\nInput (manual way): '{text_negative_manual}'")
print(f"Prediction (manual way): {prediction_negative_manual}")

# Example 3: Neutral sentiment (Slovak - manual)
text_neutral_manual = "Počasie bolo priemerné a výlet splnil očakávania." # Slovak neutral
prediction_neutral_manual = get_manual_sentiment(text_neutral_manual, model, tokenizer, device, id2label_mapping)
print(f"\nInput (manual way): '{text_neutral_manual}'")
print(f"Prediction (manual way): {prediction_neutral_manual}")

# Example 4: Another Neutral sentiment (Slovak - manual)
text_neutral_manual_2 = "Kniha bola zaujímavá, ale koniec bol trochu predvídateľný." # Slovak neutral
prediction_neutral_manual_2 = get_manual_sentiment(text_neutral_manual_2, model, tokenizer, device, id2label_mapping)
print(f"\nInput (manual way): '{text_neutral_manual_2}'")
print(f"Prediction (manual way): {prediction_neutral_manual_2}")

Loading tabularisai/multilingual-sentiment-analysis...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


--- Manual Sentiment Analysis Example ---
Model's ID to Label mapping: {0: 'Very Negative', 1: 'Negative', 2: 'Neutral', 3: 'Positive', 4: 'Very Positive'}

Input (manual way): 'Bol to skvelý zážitok, určite odporúčam každému!'
Prediction (manual way): {'label': 'Very Positive', 'score': 0.49956753849983215}

Input (manual way): 'Úplne zlé, nikdy viac sa sem nevrátim.'
Prediction (manual way): {'label': 'Negative', 'score': 0.5191943645477295}

Input (manual way): 'Počasie bolo priemerné a výlet splnil očakávania.'
Prediction (manual way): {'label': 'Neutral', 'score': 0.861337423324585}

Input (manual way): 'Kniha bola zaujímavá, ale koniec bol trochu predvídateľný.'
Prediction (manual way): {'label': 'Negative', 'score': 0.8011287450790405}

--- Final Task Confirmation ---
The code has been successfully adapted for the 'tabularisai/multilingual-sentiment-analysis' model.
It demonstrates its functionality with updated Slovak language examples, predicting 'Negative', 'Neutral', or 'Po

In [ ]:
!rm -rf ~/.cache/huggingface
print("Hugging Face cache cleared.")

Hugging Face cache cleared.
